# Week 02：最小自动驾驶汇入 MDP

States: 0 = safe gap, 1 = unsafe gap, 2 = success, 3 = failure. Actions: 0 = wait, 1 = merge. Success and failure are terminal states with zero subsequent value.

Reward for continuing to drive after waiting: -0.01; reward for successful entry: +1; reward for failed entry: -1. Waiting carries a 0.05 probability of failure per step, representing a constant risk of the lane ending (distance is not tracked). The discount factor is 0.95, and the initial state distribution is 0.5 for safe and 0.5 for unsafe.

This experiment employs dynamic programming: state values ​​are calculated based on a fixed policy, followed by greedy improvement using action values; this process iterates until the policy stabilizes.

## 1. SQLAlchemy schema


In [1]:
"""Small model-based ramp-merging MDP, adapted from week_01's domain."""
from pathlib import Path
import json
import argparse
from sqlalchemy import (MetaData, Table, Column, Integer, String, Boolean, Float,
                        ForeignKey, CheckConstraint, create_engine, select)

metadata = MetaData()
states = Table('states', metadata,
    Column('id', Integer, primary_key=True), Column('name', String, nullable=False, unique=True),
    Column('terminal', Boolean, nullable=False))
actions = Table('actions', metadata,
    Column('id', Integer, primary_key=True), Column('name', String, nullable=False, unique=True))
transitions = Table('transitions', metadata,
    Column('state_id', ForeignKey('states.id'), primary_key=True),
    Column('action_id', ForeignKey('actions.id'), primary_key=True),
    Column('next_state_id', ForeignKey('states.id'), primary_key=True),
    Column('probability', Float, nullable=False), Column('reward', Float, nullable=False),
    CheckConstraint('probability >= 0 AND probability <= 1'))
policy_actions = Table('policy_actions', metadata,
    Column('iteration', Integer, primary_key=True),
    Column('state_id', ForeignKey('states.id'), primary_key=True),
    Column('action_id', ForeignKey('actions.id'), primary_key=True),
    Column('probability', Float, nullable=False),
    CheckConstraint('probability >= 0 AND probability <= 1'))
state_values = Table('state_values', metadata,
    Column('iteration', Integer, primary_key=True),
    Column('state_id', ForeignKey('states.id'), primary_key=True),
    Column('value', Float, nullable=False))

# 0=safe gap, 1=unsafe gap, 2=success, 3=failure; 0=wait, 1=merge.
# Teaching assumptions, NOT probabilities estimated from HighwayEnv.
MODEL = {
    (0, 0): [(0, .70, -.01), (1, .25, -.01), (3, .05, -1.)],
    (1, 0): [(0, .30, -.01), (1, .65, -.01), (3, .05, -1.)],
    (0, 1): [(2, .98, 1.), (3, .02, -1.)],
    (1, 1): [(2, .20, 1.), (3, .80, -1.)],
}

def build_database(path):
    """Rebuild only this experiment's tables; reruns are deterministic."""
    engine = create_engine('sqlite:///' + Path(path).resolve().as_posix())
    with engine.connect() as conn:
        conn.exec_driver_sql('PRAGMA foreign_keys=ON')
    metadata.drop_all(engine)
    metadata.create_all(engine)
    with engine.begin() as conn:
        conn.execute(states.insert(), [dict(id=i, name=n, terminal=i >= 2)
            for i, n in enumerate(['safe_gap', 'unsafe_gap', 'success', 'failure'])])
        conn.execute(actions.insert(), [dict(id=0, name='wait'), dict(id=1, name='merge')])
        conn.execute(transitions.insert(), [dict(state_id=s, action_id=a,
            next_state_id=ns, probability=p, reward=r)
            for (s, a), outcomes in MODEL.items() for ns, p, r in outcomes])
    return engine

def load_model(engine):
    with engine.connect() as conn:
        state_rows = conn.execute(select(states)).mappings().all()
        rows = conn.execute(select(transitions)).mappings().all()
    terminal = {row['id']: row['terminal'] for row in state_rows}
    model = {}
    for row in rows:
        key = row['state_id'], row['action_id']
        model.setdefault(key, []).append((row['next_state_id'], row['probability'], row['reward']))
    for key, outcomes in model.items():
        if abs(sum(p for _, p, _ in outcomes) - 1.) > 1e-12:
            raise ValueError(f'Transitions do not sum to one: {key}')
    return model, terminal

def action_values(model, values, gamma):
    return {(s, a): sum(p * (r + gamma * values[ns]) for ns, p, r in outcomes)
            for (s, a), outcomes in model.items()}

def evaluate(model, terminal, policy, gamma=.95, tolerance=1e-10):
    if not 0 <= gamma < 1:
        raise ValueError('This implementation requires 0 <= gamma < 1')
    for s, done in terminal.items():
        if not done:
            probabilities = [policy.get((s, a), 0.) for ss, a in model if ss == s]
            if any(p < 0 or p > 1 for p in probabilities) or abs(sum(probabilities)-1) > 1e-12:
                raise ValueError(f'Invalid policy for state {s}')
    values = dict.fromkeys(terminal, 0.)
    deltas = []
    for _ in range(100000):
        q = action_values(model, values, gamma)
        updated = {s: 0. if done else sum(policy.get((s, a), 0.) * q[s, a]
            for ss, a in model if ss == s) for s, done in terminal.items()}
        delta = max(abs(updated[s] - values[s]) for s in values)
        deltas.append(delta)
        values = updated
        if delta < tolerance:
            return values, deltas
    raise RuntimeError('Policy evaluation did not converge')

def improve(model, terminal, values, gamma=.95):
    q = action_values(model, values, gamma)
    policy = {}
    for s, done in terminal.items():
        if done:
            continue
        choices = sorted(a for ss, a in model if ss == s)
        # Deterministic tie breaking avoids cycling among equivalent actions.
        best = max(choices, key=lambda a: q[s, a])
        policy.update({(s, a): float(a == best) for a in choices})
    return policy, q

def run(output_dir, gamma=.95):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    engine = build_database(output_dir / 'merge_mdp.sqlite')
    model, terminal = load_model(engine)
    policy = {(s, a): .5 for s, a in model}
    history = []
    for iteration in range(100):
        with engine.begin() as conn:
            conn.execute(policy_actions.insert(), [dict(iteration=iteration, state_id=s,
                action_id=a, probability=p) for (s, a), p in policy.items()])
            rows = conn.execute(select(policy_actions).where(
                policy_actions.c.iteration == iteration)).mappings().all()
        policy = {(r['state_id'], r['action_id']): r['probability'] for r in rows}
        values, deltas = evaluate(model, terminal, policy, gamma)
        new_policy, q = improve(model, terminal, values, gamma)
        with engine.begin() as conn:
            conn.execute(state_values.insert(), [dict(iteration=iteration, state_id=s, value=v)
                                                for s, v in values.items()])
        history.append(dict(iteration=iteration, values=values,
            policy={f'{s},{a}': p for (s, a), p in policy.items()},
            q_values={f'{s},{a}': v for (s, a), v in q.items()},
            evaluation_sweeps=len(deltas), deltas=deltas))
        if new_policy == policy:
            break
        policy = new_policy
    else:
        raise RuntimeError('Policy iteration did not converge')
    residual = max(abs(values[s] - max(q[ss, a] for ss, a in model if ss == s))
                   for s, done in terminal.items() if not done)
    report = dict(gamma=gamma, initial_distribution={'safe_gap': .5, 'unsafe_gap': .5},
        initial_return=sum(history[0]['values'][s] for s in [0, 1]) / 2,
        final_return=(values[0] + values[1]) / 2,
        bellman_optimality_residual=residual, history=history)
    (output_dir / 'results.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
    engine.dispose()
    return report



## 2. State Transition Probability

In [2]:
for (s, a), outcomes in MODEL.items():
    for ns, p, r in outcomes:
        print(f's={s}, a={a}, next={ns}, probability={p:.2f}, reward={r:.2f}')

s=0, a=0, next=0, probability=0.70, reward=-0.01
s=0, a=0, next=1, probability=0.25, reward=-0.01
s=0, a=0, next=3, probability=0.05, reward=-1.00
s=1, a=0, next=0, probability=0.30, reward=-0.01
s=1, a=0, next=1, probability=0.65, reward=-0.01
s=1, a=0, next=3, probability=0.05, reward=-1.00
s=0, a=1, next=2, probability=0.98, reward=1.00
s=0, a=1, next=3, probability=0.02, reward=-1.00
s=1, a=1, next=2, probability=0.20, reward=1.00
s=1, a=1, next=3, probability=0.80, reward=-1.00


## 3. Policy Evaluation and improvement
The initial policy selects between two actions with a probability of 0.5 in both non-terminal states.

Policy evaluation: `V_new(s) = sum_a pi(a|s) sum_next P(next|s,a) [R + gamma V(next)]`.

Action-value: `Q(s,a) = sum_next P(next|s,a) [R + gamma V(next)]`.

Policy improvement: Select the action that maximizes Q for each state. Evaluation employs synchronous updates with a convergence threshold of 1e-10; the process terminates if the greedy policy matches the current policy.

In [3]:
result = run(Path('results'))
for step in result['history']:
    print('Iteration:', step['iteration'])
    print('Policy:', step['policy'])
    print('State values:', step['values'])
    print('Action values:', step['q_values'])
    print('Evaluation sweeps:', step['evaluation_sweeps'])
    print('Final evaluation delta:', step['deltas'][-1])
    print()
print('Initial expected return:', result['initial_return'])
print('Final expected return:', result['final_return'])
print('Bellman optimality residual:', result['bellman_optimality_residual'])

Iteration: 0
Policy: {'0,0': 0.5, '0,1': 0.5, '1,0': 0.5, '1,1': 0.5}
State values: {0: 0.6121150762705412, 1: -0.35084788669242184, 2: 0.0, 3: 0.0}
Action values: {'0,0': 0.2642301526304597, '1,0': -0.10169577329546624, '0,1': 0.96, '1,1': -0.6000000000000001}
Evaluation sweeps: 27
Final evaluation delta: 9.903311504189105e-11

Iteration: 1
Policy: {'0,0': 0.0, '0,1': 1.0, '1,0': 1.0, '1,1': 0.0}
State values: {0: 0.96, 1: 0.5597385619465731, 2: 0.0, 3: 0.0}
Action values: {'0,0': 0.711837908462311, '1,0': 0.5597385620020088, '0,1': 0.96, '1,1': -0.6000000000000001}
Evaluation sweeps: 47
Final evaluation delta: 8.977441012802956e-11

Initial expected return: 0.13063359478905967
Final expected return: 0.7598692809732865
Bellman optimality residual: 5.5435656065583316e-11


## 4. Interpretation of Results and Limitations
The optimal strategy is to merge when safe and wait when unsafe. The value of merging directly from a safe state is $0.98 \times 1 + 0.02 \times (-1) = 0.96$, whereas waiting in an unsafe state allows for a potentially safer merging opportunity later.

The final value was verified using the Bellman optimality residual and cross-checked against an independent analytical solution in `test_mdp.py`. The results of this experiment should not be interpreted as reflecting the actual safety performance of vehicles; the state space and probabilities are highly simplified, the `HighwayEnv` environment was not directly executed, and the model was not learned from trajectories.

Compared to Week 1, this week retains the domain definition, success/failure rewards, and waiting costs, while condensing the five vehicle control actions into two high-level decisions. The environment generation and fixed-controller testing from last week have been replaced by policy iteration within a known MDP framework.